In [1]:
import os 
import sys

sys.path.insert(0, '..')
sys.path.insert(0, os.path.join('..', 'third_party/core_stack'))
sys.path.insert(0, os.path.join('..', 'third_party/core_stack/onroad'))

import cv2
import numpy as np
from waymo_data_wrapper import WaymoDataWrapper, convert_bbox
import nuscenes_tools.nuscenes_utils.pipelines as pipelines
from PIL import Image
from nuscenes_tools.nuscenes_utils.box3d_instance import LiDARInstance3DBoxes
from infinity.utils.vis_utils import draw_box_on_imgs, draw_map_points_on_imgs
import torchvision
from PIL import Image

import IPython
import torch 

os.environ['AWS_ENDPOINT_URL'] = 'https://idskhu5vqvtl.compat.objectstorage.us-phoenix-1.oraclecloud.com'
os.environ['AWS_ACCESS_KEY_ID'] = 'fd082c21e10475d60adc85b01be246745860a21a'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'rEW3zxhO6sV802Rg7EqgfA+CLTGh0eqHu0cIcgujauw='
os.environ['URSA_SDK_GRPC_HOSTNAME'] = 'grpc.neuron.oci.applied.dev'
os.environ['AWS_DEFAULT_REGION'] = 'us-phoenix-1'


/home/yichen-xie/miniconda3/envs/inf_stream/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/yichen-xie/miniconda3/envs/inf_stream/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2025-11-20 01:53:03.986341: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
data_config = {
    'dataset_name': 'waymo',
    # 'dataset_path': "s3://onroad-perception-datasets/exported_datasets/e2e/sandbox/samser/map_auto_labels_fcn_calib/execution_00",
    'dataset_path': "/media/research-datasets/waymo_open_dataset_v_2_0_1/validation/",
    'horizon': 4,
    'interval': 1,
    'random_interval': 0,
    'first_frame': 0,
        
    'load_bbox': True,
    'object_range': (-50, -50, -10, 50, 50, 10),

    
    'cams': ['CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT', 'CAM_SIDE_LEFT',  'CAM_SIDE_RIGHT'],
    'Ncams': 5,
    #TODO: support higher resolution and bigger model
    'input_size': (384, 672),
    'src_size': (1280, 1920),
    'keep_ratio': False,
    'mean': [0.5, 0.5, 0.5],
    'std': [0.5, 0.5, 0.5],

    # Augmentation
    'resize': (0, 0),
    'rot': (0, 0),
    'flip': False,
    'crop_h': (0.0, 0.0),
    'resize_test':0.0,
    
    'load_map': False,
    'map_sample_points_num': 30,
}

class_names = [
    'car', 'truck', 'bicycle', 'motorcycle', 'pedestrian', 'traffic_cone', 'debris', 'traffic_light', 'unknown', 'vehicle', 'cyclist', 'sign'
]
map_names = ['road_line_solid_single_white', 'road_line_broken_single_white', 'lane_surface_street', 'road_edge_sidewalk', 'road_edge_boundary', 'lane_surface_unstructure', 'crosswalk', 'road_line_solid_single_yellow']


# build augmentations
load_keys = ['img_inputs', 'gt_bboxes_3d', 'gt_labels_3d']
train_pipeline = [
    pipelines.LoadMultiViewImageFromFiles_BEVDet(data_config, is_train=True),
    pipelines.DefaultFormatBundle3D(class_names=class_names, map_names=map_names, with_gt=True, with_label=True),
    pipelines.Collect3D(keys=load_keys),
]

# build dataset
waymo_dataset = WaymoDataWrapper(data_config, train_pipeline)

In [3]:

def save_imgs(images, video_id, suffix=''):
    # Initialize video writer
    num_views = images.shape[1]
    img_h, img_w = images.shape[3:5]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use 'mp4v' for MP4 format
    video = cv2.VideoWriter(f"vis/waymo/output_video_{video_id}{suffix}.mp4", fourcc, 10, (img_w*num_views, img_h))
    
    images = torch.unbind(images, dim=0)

    # Add images to the video
    for image in images:
        frame = torch.unbind(image, dim=0)
        frame = torch.cat(frame, dim=2)
        frame = frame.permute(1, 2, 0).numpy()
        frame = (frame*255).astype(np.uint8)
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        video.write(frame)

    # Release the video writer
    video.release()


def visualize_box(sample, video_imgs):
    all_bbox_imgs = []
    curr_to_first_lidar = torch.eye(4)
    curr_to_first_lidar_list = [curr_to_first_lidar]
    for item in sample[1:]:
        meta_data = item['img_metas'].data
        curr_to_first_lidar = curr_to_first_lidar @ meta_data['curr_to_prev_lidar_rt']
        curr_to_first_lidar_list.append(curr_to_first_lidar)
    
    camera_meta_data_list = []
    for frame_id in range(len(sample)):
        item = sample[frame_id]
        meta_data = item['img_metas'].data
        batch_seq_ids = torch.zeros(1).int()  # No need to check sequence validation
        curr_to_prev_lidar = meta_data['curr_to_prev_lidar_rt']
        rot, trans, intrins, post_rot, post_trans = item['img_inputs'][1:]
        # camera_meta_datas['size'] = (img_w, img_h) # (width, height)
        
        camera2lidar = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2lidar[:, :3, :3] = rot
        camera2lidar[:, :3, 3] = trans
        lidar2camera = torch.inverse(camera2lidar)
        
        camera2image = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        camera2image[:, :3, :3] = intrins
        
        lidar2image = camera2image @ lidar2camera

        img_aug_matrix = torch.stack([torch.eye(4)]*rot.shape[0],dim=0)
        img_aug_matrix[:, :3, :3] = post_rot
        img_aug_matrix[:, :3, 3] = post_trans
        

        camera_meta_datas = {
            'rot': rot,
            'trans': trans,
            'intrins': intrins,
            'post_rot': post_rot,
            'post_trans': post_trans,
            'seq_ids': batch_seq_ids,
            'curr_to_prev_lidar': curr_to_prev_lidar,
            'img_aug_matrix': img_aug_matrix,
            'lidar2image': lidar2image,
        }


        camera_meta_data_list.append(camera_meta_datas)       

    bbox_sequence = []
    for tid in range(len(sample)):
        frame_data = sample[tid]
        gt_bboxes_3d = frame_data['gt_bboxes_3d'].data
        gt_labels_3d = frame_data['gt_labels_3d'].data

        if len(gt_labels_3d) > 0:
            xyz = gt_bboxes_3d.bottom_center
            dims = gt_bboxes_3d.dims
            yaw = gt_bboxes_3d.yaw
            first_to_seq_lidar = torch.inverse(curr_to_first_lidar_list[tid])
            
            xyz, yaw = convert_bbox(xyz, yaw, first_to_seq_lidar[:3,:3], first_to_seq_lidar[:3, 3])
            boxes = torch.cat([xyz, dims, yaw[:, None]], dim=-1)
            gt_bboxes_3d = LiDARInstance3DBoxes(boxes)
            bbox_num = boxes.shape[0]
        else:
            gt_bboxes_3d = LiDARInstance3DBoxes(torch.zeros((0, 7)))
            gt_labels_3d = torch.zeros(0)
            bbox_num = 0
        
        gt_labels_3d = gt_labels_3d.to(torch.int32)
        bbox_sequence.append([gt_bboxes_3d, gt_labels_3d, bbox_num])

    for tid in range(len(sample)):
        frame_meta_data = camera_meta_data_list[tid]
        frame_bbox = bbox_sequence[tid]
        
        frame_images = video_imgs[tid].permute(0, 2, 3, 1)
        frame_images = torch.unbind(frame_images, dim=0)
        frame_images = [Image.fromarray(img.cpu().numpy()) for img in frame_images]

        frame_data = {
            'gt_bboxes_3d': bbox_sequence[tid][0],
            'gt_labels_3d': bbox_sequence[tid][1],
            'lidar2image': camera_meta_data_list[tid]['lidar2image'],
            'img_aug_matrix': camera_meta_data_list[tid]['img_aug_matrix'],
        }
        
        bbox_imgs = draw_box_on_imgs(frame_data, frame_images, np.array(class_names))
        
        bbox_imgs = [torch.from_numpy(np.array(img)) for img in bbox_imgs]
        bbox_imgs = torch.stack(bbox_imgs, dim=0)
        bbox_imgs = bbox_imgs.permute(0, 3, 1, 2)
        all_bbox_imgs.append(bbox_imgs)
    
    return all_bbox_imgs


def save_imgs(images, i, suffix=''):
    TV3HW = torch.stack(images, dim=0)
    TV3HW = TV3HW.flatten(0, 1)
    vthw = torchvision.utils.make_grid(TV3HW, nrow=5, padding=0, pad_value=1.0)
    vthw = vthw.clone().permute(1, 2, 0).cpu().numpy()
    vthw = Image.fromarray(vthw.astype(np.uint8))
    vthw.save(f"../vis/waymo/output_video_{i}{suffix}.png")

    # # Initialize video writer
    # fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use 'mp4v' for MP4 format
    # video = cv2.VideoWriter(f"../debug/output_image_temporal_video_{i}.mp4", fourcc, 4, (img_w*num_views, img_h))

    # # Add images to the video
    # for image in video_imgs:
    #     frame = torch.unbind(image, dim=0)
    #     frame = torch.cat(frame, dim=2)
    #     frame = frame.permute(1, 2, 0).cpu().numpy()
    #     print(frame.shape, (img_w*num_views, img_h))
    #     frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    #     video.write(frame)

    # # Release the video writer
    # video.release()
    
    gif_frames = []
    for image in images:
        frame = torch.unbind(image, dim=0)
        frame = torch.cat(frame, dim=2)
        frame = frame.permute(1, 2, 0).cpu().numpy()
        gif_frames.append(Image.fromarray(frame))
    gif_frames[0].save(f"../vis/waymo/output_video_{i}{suffix}.gif", save_all=True, append_images=gif_frames[1:], duration=500, loop=0)



def get_gt_images(sample):
    gt_images = []
    for tid in range(len(sample)):
        frame_sample = sample[tid]
        frame_images = frame_sample['img_inputs'][0].clone()

        frame_images = (frame_images + 1) / 2
        frame_images = (frame_images*255).to(torch.uint8)
        
        gt_images.append(frame_images)
    return gt_images

In [4]:
waymo_iterator = iter(waymo_dataset)

for i in range(0, 100):
    print(i)
    sample = next(waymo_iterator)

    gt_images = get_gt_images(sample)
    
    gt_images_bbox = visualize_box(sample, gt_images)
    save_imgs(gt_images_bbox, i)


0
1522688014970187
1522688015070129
1522688015170074
1522688015269987
1
1553735853362214
1553735853462203
1553735853562188
1553735853662172


KeyboardInterrupt: 

In [ ]:
print(sample[0]['img_metas'].data)

tensor([[ 1.5440, -0.0233,  2.1153],
        [ 1.4965,  0.0952,  2.1155],
        [ 1.4943, -0.0966,  2.1154],
        [ 1.4319,  0.1158,  2.1157],
        [ 1.4292, -0.1158,  2.1156]])
